# Issue #09 [Data/Transform]: Variables Derivadas Vectorizadas e Indicadores de Empleabilidad

**Subentorno:** `analytics/` | **Rama:** `feat/MPHR-analytics-nosql` | **Script equivalente:** `scripts/09_variables_indicadores_empleabilidad.py`

Aquí el dataset de postulaciones se enriquece con columnas calculadas en bloque (sin recorrer fila por fila), se normalizan cargos y ciudades con diccionarios cerrados, se arma una cola de atención con `np.select()` y se calculan los indicadores del embudo de selección (`CA-ISS-09-01` a `CA-ISS-09-04`).

In [1]:
from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd
elif (cwd / 'analytics' / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd / 'analytics'
else:
    ANALYTICS_ROOT = cwd.parent

FECHAS = ['job_published_at', 'applied_at', 'updated_at']
df = pd.read_csv(ANALYTICS_ROOT / 'data' / 'processed' / 'yeb_postulaciones_preparado.csv', parse_dates=FECHAS)
FECHA_CORTE = df['updated_at'].max()
print('Dataset preparado:', df.shape, '| fecha de corte:', FECHA_CORTE.date())

Dataset preparado: (93, 27) | fecha de corte: 2026-02-02


## 1. Variables derivadas vectorizadas (`CA-ISS-09-01`)

* **`tiempo_respuesta_dias`**: días entre la postulación y el último cambio de estado, solo para postulaciones respondidas (`interview`, `accepted`, `rejected`). El dataset tabular no guarda el historial completo de estados, así que `updated_at` se usa como la fecha del paso a entrevista o del cierre.
* **`brecha_salarial_aspiracion`**: diferencia entre lo que aspira el joven y el punto medio del rango ofertado, en pesos y en porcentaje.

Para el porcentaje se usa una división segura: si el denominador fuera 0, el resultado es `NaN` y no `inf`.

In [2]:
def division_segura(num, den):
    num, den = np.asarray(num, dtype='float64'), np.asarray(den, dtype='float64')
    return np.divide(num, den, out=np.full(np.broadcast(num, den).shape, np.nan), where=(den != 0) & ~np.isnan(den))

print('division_segura(5, 0) =', division_segura([5.0], [0.0])[0])

respondida = df['status'].isin(['interview', 'accepted', 'rejected'])
df['tiempo_respuesta_dias'] = (df['updated_at'] - df['applied_at']).dt.days.where(respondida)
df['dias_sin_actualizar'] = (FECHA_CORTE - df['updated_at']).dt.days.where(df['status'].isin(['submitted', 'under_review']))

oferta_media = (df['salary_min_offered'] + df['salary_max_offered']) / 2
df['brecha_salarial_aspiracion_cop'] = df['expected_salary_cop'] - oferta_media
df['brecha_salarial_aspiracion_pct'] = np.round(division_segura(df['brecha_salarial_aspiracion_cop'], oferta_media) * 100, 2)
df['posicion_aspiracion'] = np.select(
    [df['expected_salary_cop'] < df['salary_min_offered'], df['expected_salary_cop'] > df['salary_max_offered']],
    ['Por debajo del rango', 'Por encima del rango'], default='Dentro del rango')

assert not np.isinf(df['brecha_salarial_aspiracion_pct']).any()
df[['application_id', 'status', 'tiempo_respuesta_dias', 'expected_salary_cop', 'brecha_salarial_aspiracion_pct', 'posicion_aspiracion']].head(6)

division_segura(5, 0) = nan


,application_id,status,tiempo_respuesta_dias,expected_salary_cop,brecha_salarial_aspiracion_pct,posicion_aspiracion
0,APP-2026-0001,under_review,NaN,2101373,7.76,Por encima del rango
1,APP-2026-0002,interview,1.0,1775096,-17.44,Por debajo del rango
2,APP-2026-0003,accepted,2.0,1794093,2.52,Dentro del rango
3,APP-2026-0004,rejected,3.0,2474392,-6.63,Dentro del rango
4,APP-2026-0005,submitted,NaN,2718736,6.62,Dentro del rango
5,APP-2026-0006,under_review,NaN,1609282,0.58,Dentro del rango


## 2. Normalización controlada con diccionarios y `.map()` (`CA-ISS-09-02`)

Primero se genera una clave comparable (minúsculas, sin tildes, sin espacios de más) y luego se traduce con un diccionario cerrado. Lo que no está en el diccionario **no se adivina**: queda como `SIN_EQUIVALENCIA` para que una persona decida si se agrega. Así se evita, por ejemplo, convertir "Envigado" en "Medellín" solo porque ambas son del Valle de Aburrá.

In [3]:
def clave(serie):
    sin_tildes = serie.astype('string').map(lambda s: ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c)))
    return sin_tildes.str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)

CARGO_CANONICO = {
    'analista de datos jr': 'Analista de Datos Junior', 'desarrollador backend php jr': 'Desarrollador Backend PHP Junior',
    'aprendiz sena en sistemas': 'Aprendiz SENA en Sistemas', 'agente de soporte bilingue': 'Agente de Soporte Bilingüe',
    'auxiliar de bodega e inventarios': 'Auxiliar de Bodega e Inventarios', 'asistente de marketing digital': 'Asistente de Marketing Digital',
    'auxiliar contable jr': 'Auxiliar Contable Junior', 'ejecutivo comercial juvenil': 'Ejecutivo Comercial Juvenil',
}
FAMILIA_CARGO = {
    'Analista de Datos Junior': 'Tecnología y datos', 'Desarrollador Backend PHP Junior': 'Tecnología y datos',
    'Aprendiz SENA en Sistemas': 'Tecnología y datos', 'Agente de Soporte Bilingüe': 'Servicio al cliente',
    'Auxiliar de Bodega e Inventarios': 'Logística', 'Asistente de Marketing Digital': 'Mercadeo y ventas',
    'Ejecutivo Comercial Juvenil': 'Mercadeo y ventas', 'Auxiliar Contable Junior': 'Contable y financiero',
}
CIUDAD_CANONICA = {'medellin': 'Medellín', 'bogota': 'Bogotá', 'cali': 'Cali', 'barranquilla': 'Barranquilla', 'bucaramanga': 'Bucaramanga', 'pereira': 'Pereira'}

df['cargo_normalizado'] = clave(df['job_title']).map(CARGO_CANONICO).fillna('SIN_EQUIVALENCIA')
df['familia_cargo'] = df['cargo_normalizado'].map(FAMILIA_CARGO).fillna('SIN_EQUIVALENCIA')
df['ciudad_normalizada'] = clave(df['ciudad_candidato']).map(CIUDAD_CANONICA).fillna('SIN_EQUIVALENCIA')
assert (df[['cargo_normalizado', 'familia_cargo', 'ciudad_normalizada']] != 'SIN_EQUIVALENCIA').all().all()

variantes = pd.Series(['  ANALISTA DE DATOS JR ', 'Community Manager'])
print('Prueba de variantes:', clave(variantes).map(CARGO_CANONICO).fillna('SIN_EQUIVALENCIA').tolist())
df.groupby('familia_cargo').size().rename('postulaciones').to_frame()

Prueba de variantes: ['Analista de Datos Junior', 'SIN_EQUIVALENCIA']


,postulaciones
familia_cargo,
Contable y financiero,12
Logística,12
Mercadeo y ventas,23
Servicio al cliente,10
Tecnología y datos,36


## 3. Cola de atención con `np.select()` y precedencia explícita (`CA-ISS-09-03` / `TEC-014`)

`np.select()` revisa las condiciones **en el orden en que se escriben** y asigna la primera que se cumple. Por eso el orden es la regla de negocio:

1. **P1** — inconsistencia de identidad o edad (edad fuera de 16–24, bandera de menor contradictoria, menor sin permiso, identificadores repetidos).
2. **P2** — postulación activa que lleva más días sin moverse que el percentil 90 del tiempo de respuesta.
3. **P3** — postulación activa con perfil completo (≥ 85 %), lista para entrevista.

Se suman los 4 registros que el Issue #05 dejó en cuarentena por conflicto de identificador (sin sus datos de contacto), porque son justamente los casos que deben aparecer como P1.

In [4]:
cuarentena = pd.read_csv(ANALYTICS_ROOT / 'outputs' / 'cuarentena_identificadores_conflicto.csv', parse_dates=FECHAS)
cuarentena = cuarentena.drop(columns=['email_contacto', 'telefono_contacto'])
cuarentena['tiempo_respuesta_dias'] = (cuarentena['updated_at'] - cuarentena['applied_at']).dt.days.where(cuarentena['status'].isin(['interview', 'accepted', 'rejected']))
cuarentena['dias_sin_actualizar'] = (FECHA_CORTE - cuarentena['updated_at']).dt.days.where(cuarentena['status'].isin(['submitted', 'under_review']))
cuarentena['familia_cargo'] = clave(cuarentena['job_title']).map(CARGO_CANONICO).map(FAMILIA_CARGO).fillna('SIN_EQUIVALENCIA')
cuarentena['origen'] = 'cuarentena'
df['origen'] = 'preparado'
cola = pd.concat([df, cuarentena], ignore_index=True)

UMBRAL_DIAS = int(np.ceil(df['tiempo_respuesta_dias'].quantile(0.90)))
activa = cola['status'].isin(['submitted', 'under_review'])
condiciones = [
    (~cola['candidate_age'].between(16, 24)) | (cola['is_protected_minor'] != (cola['candidate_age'] < 18))
    | (cola['is_protected_minor'] & ~cola['minor_permit_verified']) | cola.duplicated('application_id', keep=False)
    | (cola.groupby('documento_hash')['candidate_hash'].transform('nunique') > 1),
    activa & (cola['dias_sin_actualizar'] > UMBRAL_DIAS),
    activa & (cola['profile_completeness_pct'] >= 85) & (cola['familia_cargo'] != 'SIN_EQUIVALENCIA'),
]
etiquetas = ['P1 - Revisar identidad o edad', 'P2 - Postulación desactualizada', 'P3 - Apto para entrevista']
cola['prioridad_atencion'] = np.select(condiciones, etiquetas, default='Sin acción prioritaria')

print('Umbral de desactualización:', UMBRAL_DIAS, 'días')
pd.crosstab(cola['prioridad_atencion'], cola['origen'], margins=True)

Umbral de desactualización: 7 días


origen,cuarentena,preparado,All
prioridad_atencion,,,
P1 - Revisar identidad o edad,4,0,4
P2 - Postulación desactualizada,0,34,34
P3 - Apto para entrevista,0,1,1
Sin acción prioritaria,0,58,58
All,4,93,97


**¿Qué pasaría si se invierte el orden?** Se toma una postulación que cumple las tres condiciones a la vez y se clasifica con ambos órdenes:

In [5]:
caso = cola.loc[activa & (cola['profile_completeness_pct'] >= 85)].iloc[[0]].copy()
caso[['candidate_age', 'is_protected_minor', 'dias_sin_actualizar']] = [17, False, UMBRAL_DIAS + 10]
c = [
    (~caso['candidate_age'].between(16, 24)) | (caso['is_protected_minor'] != (caso['candidate_age'] < 18)),
    caso['dias_sin_actualizar'] > UMBRAL_DIAS,
    caso['profile_completeness_pct'] >= 85,
]
print('Orden correcto  (P1 > P2 > P3):', np.select(c, etiquetas, default='-')[0])
print('Orden invertido (P3 > P2 > P1):', np.select(c[::-1], etiquetas[::-1], default='-')[0])

Orden correcto  (P1 > P2 > P3): P1 - Revisar identidad o edad
Orden invertido (P3 > P2 > P1): P3 - Apto para entrevista


Con el orden invertido, un joven de 17 años registrado como mayor de edad terminaría citado a entrevista antes de revisar su identidad. Por eso el script verifica la precedencia en cada ejecución.

## 4. Indicadores del embudo (`CA-ISS-09-04` / `CP-049C`)

Cada indicador del reporte trae su numerador, su denominador y su definición, para que cualquiera pueda recalcularlo. Se comparan aquí los valores globales con un cálculo directo:

In [6]:
metricas = pd.read_csv(ANALYTICS_ROOT / 'outputs' / 'metricas_empleabilidad_yeb.csv')
glob = metricas[metricas['segmento'] == 'Global'].set_index('indicador')

st = df['status']
assert np.isclose(glob.loc['tasa_rechazo', 'valor'], 100 * (st == 'rejected').sum() / st.isin(['accepted', 'rejected']).sum(), atol=0.01)
assert np.isclose(glob.loc['tiempo_medio_respuesta', 'valor'], df['tiempo_respuesta_dias'].mean(), atol=0.01)
assert np.isclose(glob.loc['conversion_entrevista_a_contratacion', 'valor'], 100 * (st == 'accepted').sum() / st.isin(['interview', 'accepted']).sum(), atol=0.01)
glob[['numerador', 'denominador', 'valor', 'unidad']]

,numerador,denominador,valor,unidad
indicador,,,,
postulaciones_analizadas,93.00,NaN,93.00,postulaciones
tasa_avance_a_revision,73.00,93.0,78.49,%
tasa_avance_a_entrevista,37.00,93.0,39.78,%
tasa_contratacion,20.00,93.0,21.51,%
conversion_entrevista_a_contratacion,20.00,37.0,54.05,%
tasa_rechazo,18.00,38.0,47.37,%
tiempo_medio_respuesta,220.00,55.0,4.00,días
tiempo_mediano_respuesta,NaN,55.0,4.00,días
tasa_postulaciones_desactualizadas,34.00,38.0,89.47,%


In [7]:
# Comparación por cohorte de edad
filtro = metricas['segmento'].str.startswith('Cohorte') & metricas['indicador'].isin(
    ['tasa_avance_a_entrevista', 'tasa_contratacion', 'tasa_rechazo', 'tiempo_medio_respuesta'])
metricas[filtro].pivot(index='indicador', columns='segmento', values='valor')

segmento,Cohorte: Joven adulto (18-24),Cohorte: Menor protegido (16-17)
indicador,,
tasa_avance_a_entrevista,43.06,28.57
tasa_contratacion,22.22,19.05
tasa_rechazo,42.86,60.00
tiempo_medio_respuesta,3.79,4.75
